# Sionna RT channel

> A channel ray-traced beforehand with Sionna RT, loaded from numpy arrays.

In [ ]:
#| default_exp sionna_rt

In [ ]:
#| hide
from nbdev.showdoc import *

A channel ray-traced beforehand with [Sionna RT](https://nvlabs.github.io/sionna/rt/): the channel coefficients
(CFR, e.g. from `paths.cfr(...)`) between every pair of a set of positions, saved as numpy arrays:

* `H[rx, tx]`: complex, `(N, N)` or `(N, N, num_subcarriers)` (the path gain of a link is `|H|²`, averaged over the subcarriers)
* `positions[i]`: the position of index `i` (e.g. a grid cell `(x, y)`)

Nodes move, so the channel is told where they are (`set_positions`, e.g. at every step of an environment); a
transmission between two nodes then gets the SNR of their positions,
`tx_power_w (the link's) x path gain / noise_power_w`, and succeeds with the packet error rate of that SNR:
`'bpsk'` (BPSK over the SNR, so bigger packets fail more often) or `'threshold'` (received above `snr_threshold_db`).
Two nodes at the same position always communicate.

The coefficients must keep the path loss: computed with `paths.cfr(..., normalize=False)`. With
`normalize=True` every link has `|H|² = 1` (only the phases differ), so every link would be equally good; the
channel warns when it is given such an array.

In [ ]:
#| export
from __future__ import annotations

import math
import random
import warnings
from typing import Mapping, Sequence

import numpy as np

from comm_core.channel import Channel
from comm_core.core import NodeId, Transmission, TransmissionResult
from comm_core.topology import Link

In [ ]:
#| export
def bpsk_bit_error_rate(
    snr: float, # Linear signal-to-noise ratio
) -> float:
    "Bit error rate of BPSK over AWGN: Q(sqrt(2 snr))."
    return 0.5 * math.erfc(math.sqrt(max(snr, 0.0)))


def packet_error_rate(
    snr_db: float, # Signal-to-noise ratio (dB); -inf: no signal
    size_bits: int, # Packet size
    model: str = 'bpsk', # 'bpsk' (independent BPSK bit errors) or 'threshold'
    threshold_db: float = 10.0, # 'threshold': packets at or above this SNR are received
) -> float:
    "Probability that a packet is lost."
    if snr_db == math.inf:
        return 0.0
    if snr_db == -math.inf:
        return 1.0
    if model == 'threshold':
        return 0.0 if snr_db >= threshold_db else 1.0
    if model == 'bpsk':
        ber = bpsk_bit_error_rate(10 ** (snr_db / 10))
        return 1.0 - (1.0 - ber) ** max(int(size_bits), 1)
    raise ValueError(f"model: 'bpsk' or 'threshold', not {model!r}")

In [ ]:
assert packet_error_rate(30, 1000) < 1e-6 and packet_error_rate(-5, 1000) > 0.99
assert packet_error_rate(4, 10) < packet_error_rate(4, 1000)                  # bigger packets fail more often
assert packet_error_rate(9, 100, 'threshold') == 1.0 and packet_error_rate(10, 100, 'threshold') == 0.0
assert packet_error_rate(-math.inf, 8) == 1.0 and packet_error_rate(math.inf, 8) == 0.0

In [ ]:
#| export
def _key(position) -> tuple:
    return tuple(np.asarray(position).reshape(-1).tolist())


class SionnaRTChannel(Channel):
    "A channel ray-traced beforehand (Sionna RT) between a set of positions; a link gets the channel of where its nodes are."

    def __init__(
        self,
        H, # Channel coefficients `H[rx, tx]` between the positions: (N, N) or (N, N, num_subcarriers), complex
        positions: Sequence, # The N positions (e.g. grid cells (x, y)), in the order of H's indices
        noise_power_w: float = 1e-13, # Noise power at the receiver, W (1e-13 W = -100 dBm)
        per_model: str = 'bpsk', # Packet error rate of the SNR: 'bpsk' or 'threshold' (see `packet_error_rate`)
        snr_threshold_db: float = 10.0, # The 'threshold' model's SNR
        node_positions: Mapping | None = None, # Where the nodes are to begin with (see `set_positions`)
        seed: int | None = None, # Seed of the packet losses
    ):
        H = np.asarray(H)
        if H.ndim == 3:
            gain = (np.abs(H) ** 2).mean(axis=-1)
        elif H.ndim == 2:
            gain = np.abs(H) ** 2
        else:
            raise ValueError(f"H: (N, N) or (N, N, num_subcarriers), not {H.shape}")
        if gain.shape[0] != gain.shape[1] or gain.shape[0] != len(positions):
            raise ValueError(f"H is {H.shape[:2]} for {len(positions)} positions")
        self.gain = gain.astype(np.float64)
        self.positions = [_key(p) for p in positions]
        self.index = {p: i for i, p in enumerate(self.positions)}
        off = ~np.eye(len(self.positions), dtype=bool)
        if off.any() and np.allclose(self.gain[off], 1.0):
            warnings.warn("every link has |H|^2 = 1: the coefficients were normalized (paths.cfr(normalize=True)), "
                          "so the path loss is lost and every link is equally good; use normalize=False")
        self.noise_power_w = noise_power_w
        self.per_model = per_model
        self.snr_threshold_db = snr_threshold_db
        self.node_positions: dict = {}
        self.set_positions(node_positions or {})
        self.rng = random.Random(seed)

    @classmethod
    def from_files(
        cls,
        channel_path: str, # .npy of H[rx, tx]
        positions_path: str, # .npy of the positions
        **kwargs, # The other arguments of `SionnaRTChannel`
    ) -> SionnaRTChannel:
        "Load the arrays saved after ray tracing."
        return cls(np.load(channel_path), np.load(positions_path), **kwargs)

    def set_positions(
        self,
        positions: Mapping[NodeId, Sequence], # Position of (some of) the nodes, e.g. {'agent_0': (3, 5)}
    ) -> None:
        "Where the nodes are now."
        for node, p in positions.items():
            self.node_positions[node] = _key(p)

    def reset(self, seed: int | None = None) -> None:
        "`seed` reseeds the packet losses."
        if seed is not None:
            self.rng.seed(seed)

    def _index(self, node: NodeId) -> int:
        if node not in self.node_positions:
            raise KeyError(f"no position for node {node!r}: call set_positions first")
        p = self.node_positions[node]
        if p not in self.index:
            raise KeyError(f"node {node!r} is at {p}, where no channel was ray-traced ({len(self.index)} positions)")
        return self.index[p]

    def path_gain(self, sender: NodeId, receiver: NodeId) -> float:
        "Linear path gain |H|^2 from `sender` to `receiver` at their current positions (inf if co-located)."
        tx, rx = self._index(sender), self._index(receiver)
        return math.inf if tx == rx else float(self.gain[rx, tx])

    def snr_db(self, sender: NodeId, receiver: NodeId, tx_power_w: float = 1.0) -> float:
        "SNR of a transmission from `sender` to `receiver` (dB)."
        snr = tx_power_w * self.path_gain(sender, receiver) / self.noise_power_w
        return 10 * math.log10(snr) if snr > 0 else -math.inf

    def transmit(
        self,
        transmission: Transmission,
        link: Link,
    ) -> TransmissionResult:

        message = transmission.message
        gain = self.path_gain(transmission.sender, transmission.receiver)
        snr_db = self.snr_db(transmission.sender, transmission.receiver, link.tx_power_w)
        per = packet_error_rate(snr_db, message.size_bits or 1, self.per_model, self.snr_threshold_db)
        success = self.rng.random() >= per

        return TransmissionResult(
            success=success,
            transmission=transmission,
            payload=message.payload if success else None,
            latency=0.0,
            size_bits=message.size_bits or 0,
            snr_db=snr_db,
            error_rate=per,
            metadata={'path_gain_db': 10 * math.log10(gain) if gain > 0 else -math.inf},
        )

In [ ]:
show_doc(SionnaRTChannel.from_files)

In [ ]:
show_doc(SionnaRTChannel.set_positions)

In [ ]:
show_doc(SionnaRTChannel.path_gain)

In [ ]:
show_doc(SionnaRTChannel.snr_db)

### Tests

In [ ]:
from comm_core.core import Message

# 3 positions on a line: (0, 0) - (1, 0) - (5, 0); path gain falls with the distance; H[rx, tx]
positions = [(0, 0), (1, 0), (5, 0)]
gain_db = np.array([[0, -60, -95], [-60, 0, -90], [-95, -90, 0]], float)
H = np.sqrt(10 ** (gain_db / 10)) * np.exp(1j * np.arange(9).reshape(3, 3))
np.fill_diagonal(H, 0)
ch = SionnaRTChannel(H, positions, noise_power_w=1e-13, node_positions={'a': (0, 0), 'b': (1, 0), 'c': (5, 0)}, seed=0)
assert abs(ch.snr_db('a', 'b') - 70) < 1e-9 and abs(ch.snr_db('a', 'c') - 35) < 1e-9

def send(ch, sender, receiver, bits=256, n=200):
    t = Transmission(message=Message(sender, receiver, payload=1, size_bits=bits), sender=sender, receiver=receiver, start_time=0.0)
    results = [ch.transmit(t, Link(sender, receiver, tx_power_w=1e-4)) for _ in range(n)]    # 0.1 mW
    return sum(r.success for r in results) / n, results[0]

near, r = send(ch, 'a', 'b')                               # 0.1 mW x -60 dB / -100 dBm: 30 dB, always received
far, _ = send(ch, 'a', 'c')                                # -5 dB: lost
assert near == 1.0 and far == 0.0 and abs(r.snr_db - 30) < 1e-9 and abs(r.metadata['path_gain_db'] + 60) < 1e-9
# the nodes move: the channel follows
ch.set_positions({'c': (1, 0)})
assert send(ch, 'a', 'c')[0] == 1.0
ch.set_positions({'b': (0, 0)})
assert ch.path_gain('a', 'b') == math.inf and send(ch, 'a', 'b')[0] == 1.0        # same cell
# in between: losses, reproducible with the seed
ch = SionnaRTChannel(H, positions, noise_power_w=2e-11, node_positions={'a': (0, 0), 'b': (1, 0)})   # 7 dB
ch.reset(seed=3); first = [send(ch, 'a', 'b', bits=1000, n=50)[0] for _ in range(3)]
ch.reset(seed=3); assert [send(ch, 'a', 'b', bits=1000, n=50)[0] for _ in range(3)] == first
assert 0 < first[0] < 1
# subcarriers, the threshold model, errors, files, normalized arrays
H3 = np.repeat(H[..., None], 4, axis=-1)
assert np.allclose(SionnaRTChannel(H3, positions).gain, ch.gain)
ch = SionnaRTChannel(H, positions, per_model='threshold', snr_threshold_db=-10, node_positions={'a': (0, 0), 'c': (5, 0)})
assert send(ch, 'a', 'c')[0] == 1.0                                            # -5 dB, above the threshold
ch.set_positions({'c': (9, 9)})
try:
    send(ch, 'a', 'c'); raise AssertionError('no error')
except KeyError as e:
    assert 'no channel was ray-traced' in str(e)
import tempfile, os
with tempfile.TemporaryDirectory() as tmp:
    np.save(os.path.join(tmp, 'H.npy'), H); np.save(os.path.join(tmp, 'pos.npy'), np.array(positions))
    ch = SionnaRTChannel.from_files(os.path.join(tmp, 'H.npy'), os.path.join(tmp, 'pos.npy'))
    assert ch.positions == [(0, 0), (1, 0), (5, 0)]
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    SionnaRTChannel(np.exp(1j * np.ones((3, 3))), positions)
    assert any('normalize' in str(x.message) for x in w)

In a `Network`, the application moves the nodes before each step:

In [ ]:
from comm_core import Network, Node, FullMeshTopology, IdealProtocol, AnalyticalBackend

ch = SionnaRTChannel(H, positions, noise_power_w=1e-8, seed=0)   # 1 W links: 20 dB at (1, 0), -15 dB at (5, 0)
net = Network([Node('a'), Node('b')], FullMeshTopology(['a', 'b']), IdealProtocol(), AnalyticalBackend(ch))
received = []
for pos_b in [(1, 0), (5, 0), (1, 0)]:                  # b walks away and comes back
    ch.set_positions({'a': (0, 0), 'b': pos_b})
    net.send(Message('a', 'b', payload='hi', size_bits=1024))
    net.step()
    received.append(len(net.receive('b')))
assert received == [1, 0, 1]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()